In [15]:
!pip install spotipy
!pip install pytz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.9/507.9 kB 9.9 MB/s eta 0:00:008.6 MB/s eta 0:00:01


In [ ]:
import spotipy
import pytz
import time
import json
from datetime import datetime, timedelta
from spotipy.oauth2 import SpotifyOAuth

# Set up Spotify API credentials
CLIENT_ID = ''  # Replace with your Client ID
CLIENT_SECRET = ''  # Replace with your Client Secret
REDIRECT_URI = 'http://localhost:8080/callback'  # Redirect URI


In [48]:
# TOP TRACKS 
# Authenticate with Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope='user-top-read'  # Scopes define the permissions you request
))

time_range = 'long_term'
all_tracks = []
max_tracks = 100  # Spotify API returns a maximum of 100 top tracks
offset = 0

while True:
    try:
        top_tracks = sp.current_user_top_tracks(limit=50, offset=offset, time_range=time_range)
        
        if not top_tracks["items"]:
            print("No hay más canciones.")
            break
            
        all_tracks.extend(top_tracks["items"])
        offset += 50
        
        # Imprime el número total de canciones obtenidas hasta ahora
        print(f"Obtenidas {len(top_tracks['items'])} canciones. Total acumulado: {len(all_tracks)}")
        
        if len(top_tracks["items"]) < 50 or len(all_tracks) >= max_tracks:
            break
        
        # Espera para evitar rate limits
        time.sleep(1)
        
    except Exception as e:
        print(f"Error: {e}")
        break
print('finished')
# Prepara el JSON
output_data = [
    {
        "playedAt": item["album"]["release_date"],
        "artistName": item["artists"][0]["name"],
        "trackName": item["name"],
        "msPlayed": item["duration_ms"]
    }
    for item in all_tracks
]

# Save to JSON file
with open("tracks_last_year.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"Successfully saved {len(output_data)} tracks to {filename}")

Obtenidas 50 canciones. Total acumulado: 50
Obtenidas 50 canciones. Total acumulado: 100
finished


In [49]:
# LASTED TRACKS
# Autenticación
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope="user-read-recently-played"
))

# Calcula el timestamp de hace 30 días
now = datetime.now(pytz.utc)
one_year_ago = now - timedelta(days=30)
timestamp_x_days_ago = int(one_year_ago.timestamp()) * 1000

# Almacena todas las canciones
all_tracks = []
current_timestamp = None  # Usado para paginación
max_requests = 20  # Límite de solicitudes para evitar bucles infinitos
request_count = 0

while request_count < max_requests:
    try:
        # Parámetros de la solicitud
        params = {"limit": 50}
        
        if current_timestamp:
            params["before"] = current_timestamp
        else:
            params["after"] = timestamp_x_days_ago
        
        # Obtiene las canciones
        recently_played = sp.current_user_recently_played(**params)
        
        # Si no hay más canciones, termina el loop
        if not recently_played["items"]:
            print("No hay más canciones.")
            break
            
        # Agrega las canciones a la lista
        all_tracks.extend(recently_played["items"])
        print(f"Obtenidas {len(recently_played['items'])} canciones. Total acumulado: {len(all_tracks)}")
        
        # Actualiza el timestamp para la próxima página
        current_timestamp = recently_played["cursors"]["before"]
        request_count += 1
        
        # Espera para evitar rate limits
        time.sleep(1)
        
    except Exception as e:
        print(f"Error: {e}")
        break

# Prepara el JSON
output_data = [
    {
        "playedAt": datetime.strptime(item["played_at"], "%Y-%m-%dT%H:%M:%S.%fZ").strftime("%Y-%m-%d %H:%M:%S"),
        "artistName": item["track"]["artists"][0]["name"],
        "trackName": item["track"]["name"],
        "msPlayed": item["track"]["duration_ms"]
    }
    for item in all_tracks
]

# Guarda en JSON
with open("recently_played.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"¡Listo! Se guardaron {len(output_data)} canciones.")